In [ ]:
import torch
import shutil
import json
import yaml
import random
from ultralytics import YOLO
from pathlib import Path
from collections import Counter

In [33]:
# 디바이스 설정
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.xpu.is_available():
    DEVICE = torch.device('xpu')
else:
    DEVICE = torch.device('cpu')

print(DEVICE)

cuda


In [34]:
# 데이터 경로 설정
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / 'data' / '이안류'
TRAIN_DIR = DATA_DIR / 'Training'
VAL_DIR = DATA_DIR / 'Validation'
YOLO_DIR = DATA_DIR / 'Offshore_YOLO'

TRAIN_IMAGE = TRAIN_DIR / '01.원천데이터'
TRAIN_LABEL = TRAIN_DIR / '02.라벨링데이터'

VAL_IMAGE = VAL_DIR / '01.원천데이터'
VAL_LABEL = VAL_DIR / '02.라벨링데이터'

In [35]:
# Offshore_YOLO 내 폴더 생성
for sub in[
    "images/train", "images/val", "images/test",
    "labels/train", "labels/val", "labels/test"
]:
    (YOLO_DIR/sub).mkdir(parents=True, exist_ok=True)

In [36]:
# JSON 형식 확인
'''
{
    "image_info": {
        "file_name": "HD_GLORY_20190607_090204.jpg",
        "collection_method": "Null",
        "ID_code": "HD_GLORY",
        "date": "Null",
        "GPS": "35.1594,129.1586",
        "resolution": "1920,1080",
        "focus_distance": "Null",
        "model_name": "Null",
        "image_size": "Null"
    },
    "annotations": {
        "bounding_count": 1, # 0  # 2
        "class": 1, # 0  # 1
        "details": "Cloudy",
        "sun_light": "Null",
        "object": "Yes",
        "drawing": [ # null
            [
                [
                    1001,
                    370
                ],
                [
                    1281,
                    370
                ],
                [
                    1281,
                    528
                ],
                [
                    1001,
                    528
                ]
            ]
        ],
        "rip_current_duration": null,
        "rip_current_phase": null,
        "significant_wave_height": 1.2,
        "significant_wave_period": 6.61,
        "wind_velocity": 2.6,
        "wind_direction": 65.0,
        "wave_direction_sprading_factor": 41.7388,
        "spectrum_spreading_factor": 2.9176,
        "peak_period": 6.2439,
        "peak_direction": 173.0,
        "angle_of_incidence_of_the_beach": 178,
        "tide_level": 1.03
    }
}
'''

# 바운딩 박스 좌표가 annotaions의 drawing 안에 들어있음

'\n{\n    "image_info": {\n        "file_name": "HD_GLORY_20190607_090204.jpg",\n        "collection_method": "Null",\n        "ID_code": "HD_GLORY",\n        "date": "Null",\n        "GPS": "35.1594,129.1586",\n        "resolution": "1920,1080",\n        "focus_distance": "Null",\n        "model_name": "Null",\n        "image_size": "Null"\n    },\n    "annotations": {\n        "bounding_count": 1, # 0  # 2\n        "class": 1, # 0  # 1\n        "details": "Cloudy",\n        "sun_light": "Null",\n        "object": "Yes",\n        "drawing": [ # null\n            [\n                [\n                    1001,\n                    370\n                ],\n                [\n                    1281,\n                    370\n                ],\n                [\n                    1281,\n                    528\n                ],\n                [\n                    1001,\n                    528\n                ]\n            ]\n        ],\n        "rip_current_duration": null,

In [37]:
# yolo로 변환하기
# 전부 원본 이미지 크기로 정규화 필요
# yolo : class, x_center, y_center, width, height (바운딩 박스의 너비, 높이)
def json_to_yolo(file_name):
    drawing = file_name['annotations']['drawing']
    class_name = 0 # 정상일 때는 drawing이 None이므로 이안류 발생 시만 저장
    img_width,img_height = file_name['image_info']['resolution'].split(',')

    img_width, img_height = int(img_width), int(img_height)

    yolo = []

    if drawing is not None:

        for draw in drawing:
            min_x = draw[0][0]
            max_x = draw[0][0]
            min_y = draw[0][1]
            max_y = draw[0][1]

            for x,y in draw:
                if x < min_x :
                    min_x = x

                if x > max_x:
                    max_x = x

                if y < min_y:
                    min_y = y

                if y > max_y:
                    max_y = y

            width = (max_x - min_x) / img_width
            height = (max_y - min_y) / img_height

            x_center = (min_x + max_x) /2 / img_width
            y_center = (min_y + max_y) /2 / img_height

            yolo.append([class_name,x_center,y_center,width,height])


        # 정상일 때는 빈 리스트 반환

    return yolo        


In [38]:
def conv_json_yolo(json_path, label_path):
    file_name = json.loads(json_path.read_text(encoding='utf-8'))
    yolo_anns = json_to_yolo(file_name)

    # join을 위한 문자열 변환
    lines = []
    for ann in yolo_anns:
        c_id, x, y, w, h = ann
        lines.append(f"{c_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")

    label_path.parent.mkdir(parents=True, exist_ok=True)
    label_path.write_text("\n".join(lines), encoding="utf-8")
        

In [ ]:
# 데이터 셋 자체에서 train/validation으로 나누어져있으나 같은 장소 각도가 나뉘어져 존재
# 과적합 방지를 위해 새롭게 데이터를 분리해야 함

# 카메라(장소) 단위로 train/val/test를 완전히 분리
# -> 같은 카메라(배경)가 train과 val/test에 동시에 들어가면
#    모델이 이안류가 아니라 배경(카메라별 장면)을 외워 과적합하는 문제가 있었음
# 폴더 이름: TS_1.이미지_1.해운대_3.PARA2
# 사진 이름: HD_PARA2_20190630_090623
# 해수욕장, 해수욕장 번호, 카메라 번호, 카메라 코드
# HD, 1, 3, PARA2

def prepare_split(): # train, TS_1.이미지_1.해운대_3.PARA2
    SITES = [
        ("해운대",1,1,"GLORY"), ("해운대",1,2,"PARA1"),
        ("해운대",1,3,"PARA2"), ("해운대",1,4,"SEAC1"),
        ("송정",2,1,"WHIB1"), ("송정",2,2,"SJHT1"),
        ("중문",3,1,"BADA1"), ("중문",3,2,"BADA2"),
        ("대천",4,1,"MUDCH"), ("대천",4,2,"ZIPTR"),
        ("낙산",5,1,"NSBE1"), ("낙산",5,2,"NSBE2"),
    ]

    SEED = 2026

    # 카메라 단위로 셔플 후 분할 (같은 카메라가 여러 split에 들어가지 않도록)
    sites_shuffled = SITES.copy()
    random.Random(SEED).shuffle(sites_shuffled)

    n = len(sites_shuffled)
    n_train = round(n * 0.6)
    n_val   = round(n * 0.2)

    train_sites = sites_shuffled[:n_train]
    val_sites   = sites_shuffled[n_train : n_train + n_val]
    test_sites  = sites_shuffled[n_train + n_val :]

    site_to_split = {}
    for _, _, _, camera_id in train_sites: site_to_split[camera_id] = "train"
    for _, _, _, camera_id in val_sites:   site_to_split[camera_id] = "val"
    for _, _, _, camera_id in test_sites:  site_to_split[camera_id] = "test"

    print("train sites:", [s[3] for s in train_sites])
    print("val sites:  ", [s[3] for s in val_sites])
    print("test sites: ", [s[3] for s in test_sites])

    records = []

    for site in SITES:
        beach, beach_num, camera_num, camera_id = site
        img_train_name = f'TS_1.이미지_{beach_num}.{beach}_{camera_num}.{camera_id}'
        label_train_name = f'TL_1.JSON_{beach_num}.{beach}_{camera_num}.{camera_id}'
        img_val_name = f'VS_1.이미지_{beach_num}.{beach}_{camera_num}.{camera_id}'
        label_val_name = f'VL_1.JSON_{beach_num}.{beach}_{camera_num}.{camera_id}'


        for img_root, label_root, img_name, label_name in [
        (TRAIN_IMAGE, TRAIN_LABEL, img_train_name, label_train_name),
        (VAL_IMAGE,   VAL_LABEL,   img_val_name,   label_val_name),]:
            
            # 실제 dir 경로
            img_dir = img_root / img_name
            label_dir = label_root / label_name

            for image_path in img_dir.glob("*.jpg"):
                beach_code, camera_code, date_str, time_str = image_path.stem.split('_') # [HD,PARA2,20190630,090623]

                img_label = {"image_path":image_path, "json_path":label_dir / f"{image_path.stem}.json", "site":camera_code, "date":date_str, "hour":time_str[:2], "split":site_to_split[camera_code]}
                records.append(img_label)


    for r in records:
        out_image_dir = YOLO_DIR / "images" / r["split"]
        out_label_dir = YOLO_DIR / "labels" / r["split"]

        shutil.copy2(r["image_path"], out_image_dir / r["image_path"].name)
        conv_json_yolo(r["json_path"], out_label_dir / f'{r["image_path"].stem}.txt')


    return records


In [43]:
records = prepare_split()

In [44]:
# 데이터 출력
Counter(r["split"] for r in records)
Counter((r["site"], r["split"]) for r in records)

Counter({('SJHT1', 'train'): 38726,
         ('BADA1', 'train'): 23417,
         ('BADA2', 'train'): 19038,
         ('PARA1', 'val'): 17840,
         ('BADA1', 'val'): 17109,
         ('PARA2', 'val'): 16749,
         ('SEAC1', 'train'): 14539,
         ('GLORY', 'val'): 14367,
         ('BADA2', 'test'): 11329,
         ('SJHT1', 'test'): 11293,
         ('PARA1', 'test'): 9459,
         ('GLORY', 'train'): 8323,
         ('PARA1', 'train'): 8113,
         ('WHIB1', 'train'): 8111,
         ('SJHT1', 'val'): 7441,
         ('NSBE2', 'val'): 7288,
         ('PARA2', 'train'): 6594,
         ('WHIB1', 'test'): 6041,
         ('MUDCH', 'train'): 5340,
         ('NSBE1', 'val'): 5021,
         ('NSBE1', 'train'): 3901,
         ('NSBE2', 'train'): 3768,
         ('ZIPTR', 'train'): 2646,
         ('MUDCH', 'val'): 2286,
         ('MUDCH', 'test'): 1977,
         ('BADA2', 'val'): 951,
         ('PARA2', 'test'): 944,
         ('NSBE2', 'test'): 915,
         ('GLORY', 'test'): 895,
     

In [46]:
# yaml 위치 지정
custom_offshore_yaml = YOLO_DIR / "custom_offshore.yaml"

In [47]:
data_config = {
    "path" : str(YOLO_DIR.resolve()),
    "train" : "images/train",
    "val" : "images/val",
    "test" : "images/test",
    "names" : {0:'OFFSHORE'}
}

with open(custom_offshore_yaml, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_config, f, allow_unicode=True, sort_keys=False)

print("생성된 YAML 경로: ", custom_offshore_yaml)
print(custom_offshore_yaml.read_text(encoding='utf-8'))

생성된 YAML 경로:  d:\haewoon\KDT\11_CV\data\이안류\Offshore_YOLO\custom_offshore.yaml
path: D:\haewoon\KDT\11_CV\data\이안류\Offshore_YOLO
train: images/train
val: images/val
test: images/test
names:
  0: OFFSHORE



In [48]:
model = YOLO("yolo11s.pt")
model

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(96, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(128, eps=0.001, momentu

In [ ]:
results = model.train(
    data=str(custom_offshore_yaml),
    epochs=10,
    imgsz=640,
    batch=16,
    device=DEVICE,
    workers=2,
    project=str(PROJECT_DIR / "runs" / "detect"),
    name="Offshore_yolo11s",
    exist_ok=True,
    seed=2026
)

New https://pypi.org/project/ultralytics/8.4.149 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.146  Python-3.11.7 torch-2.14.0+cu132 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=d:\haewoon\KDT\11_CV\data\\Offshore_YOLO\custom_offshore.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det

In [ ]:
best_path = Path('runs/detect/Offshore_yolo11s/weights/best.pt')
model = YOLO(str(best_path))

In [ ]:
val_results = model.val(
    data=str(custom_offshore_yaml),
    split='val',
    imgsz=640,
    batch=16,
    device=DEVICE,
    workers=2
)

In [ ]:
print("mAP50: ",val_results.box.map50)
print("mAP50-95: ", val_results.box.map)
print('mAP75: ', val_results.box.map75)

In [ ]:
test_results = model.val(
    data=str(custom_offshore_yaml),
    split='test',
    imgsz=640,
    batch=16,
    device=DEVICE,
    workers=2
)

In [ ]:
print("mAP50: ",test_results.box.map50)
print("mAP50-95: ", test_results.box.map)
print('mAP75: ', test_results.box.map75)

In [ ]:
# source_dir = YOLO_DIR / 'images' / 'test'
# pred_results = model.predict(
#     source=str(source_dir),
#     imgsz=640,
#     conf=0.30,
#     device=DEVICE,
#     save=True,
#     save_txt=True,
#     save_conf=True,
#     name='offshore_predict',
#     exist_ok=True
# )

# print(f'예측 이미지 수: {len(pred_results)}')